# Webページから情報を取り出してみよう

きょうは **スクレイピング** に挑戦します。
スクレイピング＝Webページを読みこんで、ほしい部分だけを取り出すこと。

材料はこの2つだけ。むずかしい準備はいりません。

- `requests` … ページを取ってくる係
- `BeautifulSoup` … 取ってきたページをバラバラに分解して、探しやすくする係


### 講師メモ：全体の流れ

**50分×2コマ。** 1コマ目はステップ2まで、2コマ目でステップ3以降。

| コマ | ステップ | 内容 | 目安 |
|---|---|---|---|
| 1 | 0 | ブラウザでサイトを見る | 4分 |
| 1 | 1 | ページのタイトルを取る | 11分 |
| 1 | 2 | 商品名を全部並べる | 14分 |
| 2 | 3 | 価格を取って最安値を探す | 12分 |
| 2 | 4 | ほしいものだけ選ぶ（if・並べかえ） | 10分 |
| 2 | 5 | カテゴリごとに数える（辞書・平均） | 8分 |
| 2 | 6 | 詳細ページを辿る | 8分 |
| 2 | 7 | 本物の Wikipedia 記事を読む（発展） | 時間が余ったとき |

- **1コマ目はステップ2で切る。** 値段には入らない。中途半端に切ると2コマ目の頭で説明をやり直すことになる。
- 2コマ目の頭に、いちばん上から実行しなおす時間を4分置く。
  休憩でランタイムが切れて変数が消えているため。**これは故障ではないと最初に言う。**
- ステップ7だけが発展。早く終わった生徒の受け皿。
- 各コードセルは上から順に実行する前提。飛ばすと `NameError` になる。
- **外部にアクセスするのは 7-5 の1セルだけ**（講師機で1回）。それ以外は自前のサイトで完結する。


## 準備

最初に1回だけ実行するセルです。
左の ▶ ボタンを押すか、`Shift + Enter` を押すと実行できます。


In [ ]:
import requests
from bs4 import BeautifulSoup

# 練習用サイトのアドレス。これ以降このURLを直接書くことはありません
BASE_URL = "https://yuracode.github.io/isshop/"

print("準備できました")

### 講師メモ

- Colab には `requests` も `bs4` も最初から入っている。`pip install` は不要。
- `import` を忘れると以降すべて `NameError`。実行し忘れが最も多い事故なので、
  ここで全員の画面に「準備できました」が出たことを目視で確認する。
- URL を書くのはこのセルだけ。以降は `BASE_URL` を使い回す。


## ステップ0：まず相手を見る

コードを書く前に、これから読みこむページをブラウザで開いてみましょう。

https://yuracode.github.io/isshop/

「サイバー購買部」というお店のページです。
パン・軽食・飲み物・文具・日用品の5つのグループに、ぜんぶで30商品ならんでいます。
このページから、商品名や値段をプログラムで取り出すのが今日のゴールです。

### 講師メモ

- ここではコードを書かせない。**相手を見てから触る**という順番を体で覚えてもらう。
- 余裕があれば右クリック →「検証」で HTML を見せる。
  同じ形の `<div class="item">` がくり返しならんでいることだけ指させれば十分。
  タグの文法説明には入らないこと。
- カテゴリごとに `<section>` で区切ってあるが、この区切りは今日は使わない。
  `find_all` は section をまたいで30個ぜんぶ拾う、とだけ押さえておく。


## ステップ1：ページのタイトルを取る

まずは小さく成功しましょう。ページの「タイトル」だけを取り出します。

やることは2つ。**ページを取ってくる** → **タイトルを探す**。


### 1-1. ページを取ってくる

`requests.get(...)` で、指定したアドレスのページを取ってきます。


In [ ]:
res = requests.get(BASE_URL)
res.encoding = "utf-8"   # 日本語が文字化けしないようにするおまじない
html = res.text

### 1-2. 取れたものを見る

`html` の中身を、最初の300文字だけのぞいてみましょう。


In [ ]:
print(html[:300])

記号だらけで読みにくいですね。これが HTML です。

このままでは探しにくいので、`BeautifulSoup` に渡して**分解**してもらいます。
分解しておくと「タイトルを持ってきて」といった頼み方ができるようになります。


In [ ]:
soup = BeautifulSoup(html, "html.parser")

### 1-3. タイトルを取り出す

`soup.title` でタイトルの部分、`.text` でその中の文字だけが取れます。


In [ ]:
print(soup.title.text)

### 講師メモ

- `res.encoding = "utf-8"` は文字化け防止。生徒には「おまじない」で流してよい。
  聞かれたら「日本語をどう読むかをこちらから指定している」とだけ答える。
- `soup.title` だけだと `<title>サイバー購買部</title>` とタグごと出る。
  `.text` を付けると中身だけになる、という差をその場で見せると理解が早い。
- 「取れた！」を必ず声に出して確認する。ここが今日最初の成功体験。


### やってみよう

`title` を `h1` に変えると、ページの大きな見出しが取れます。


In [ ]:
print(soup.h1.text)

## ステップ2：商品名を全部ならべる

つぎは商品名です。30個あるので、**まとめて全部**取ります。

このサイトでは、商品1つ分が `<div class="item">` という箱に入っています。
`find_all` を使うと、その箱を**全部**まとめて取ってこられます。

In [ ]:
items = soup.find_all("div", class_="item")

いくつ取れたか数えてみましょう。`len(...)` は個数を数える命令です。


In [ ]:
print(len(items))

### 講師メモ

- `30` が出れば成功。ここで数が合わないときは `class_` のつづり間違いがほぼ100%。
- `find` は最初の1つだけ、`find_all` は全部。ここは板書して区別させる。
- `class_` のアンダースコアを落とすとエラーになる。Python の予約語 `class` を
  避けるための決まりごと、と一言だけ添える（詳しい説明には踏みこまない）。


### 2-2. 箱の中から名前を取り出す

`items` の中身を1つずつ順番に見ていきます。これが**くり返し（ループ）**です。

商品名は `<h2 class="item-name">` に入っています。


In [ ]:
for item in items:
    name = item.find("h2", class_="item-name")
    print(name.text)

### 講師メモ

- 30行ずらっと出た瞬間がいちばん盛り上がるところ。手を止めて画面を見せ合わせる。
  「30個ぜんぶ手で打つ気になる？」と一言添えると、ループの価値が伝わる。
- `for` の下の行が字下げ（インデント）されている点に触れる。
  全角スペースが混ざると `IndentationError`。発生したら全員に注意喚起する。


### やってみよう

`items[0]` は「1つめの商品」という意味です。
`0` を `1` や `2` に変えて、別の商品の名前を出してみましょう。


In [ ]:
print(items[0].find("h2", class_="item-name").text)

## 1コマ目はここまで

30個の商品名が、一瞬でならびました。ここで1コマ目は終わりです。

**「ファイル」→「保存」（`Ctrl + S`）で保存**してから閉じてください。


## 2コマ目のはじめに

休憩のあいだに、Colab とのつながりは切れています。
**いちばん上の「準備」のセルから、もう一度順番に ▶ を押してください。**

ステップ2まで実行すれば `items` がまた使えるようになります。すぐ終わります。


## ステップ3：値段も取って、最安値を探す

名前が取れたので、つぎは値段です。
値段は `<span class="item-price">` に、数字だけが入っています。


In [ ]:
for item in items:
    name = item.find("h2", class_="item-name").text
    price = item.find("span", class_="item-price").text
    print(name, price)

### 3-2. 文字を数字に変える

いま取れた `150` は、じつは**文字**であって数ではありません。
文字のままだと大小をくらべられないので、`int(...)` で数に変えます。


In [ ]:
price_text = items[0].find("span", class_="item-price").text
price = int(price_text)

print(price_text, "→", price)

### 講師メモ

- 画面上はどちらも `150` に見えるので、差が伝わりにくい。
  `print(price_text + 10)` を実演してエラーを見せると腹落ちする（文字と数はたし算できない）。
- HTML 側であえて「150円」ではなく「150」だけを入れてある。
  文字列処理の話に脱線させないための設計、と押さえておく。


### 3-3. 一番安い商品を探す

順番に見ていって、「いままでで一番安い値段」より安ければ覚えなおす。
これをくり返せば、最後に残るのが最安値です。


In [ ]:
cheapest_name = ""
cheapest_price = 99999

for item in items:
    name = item.find("h2", class_="item-name").text
    price = int(item.find("span", class_="item-price").text)
    if price < cheapest_price:
        cheapest_price = price
        cheapest_name = name

In [ ]:
print("一番安いのは", cheapest_name, "で", cheapest_price, "円です")

### 講師メモ

- `cheapest_price = 99999` の意味（最初はありえない大きな数を置く）を口頭で補う。
- ここまでで49分程度（冒頭の HTML の話8分を含む）。**授業としてはここで完結してよい。**
- 早く終わった生徒にはステップ4へ進ませ、そうでない生徒はやってみようを触らせる。


### やってみよう

`<` を `>` に、`99999` を `0` に変えると、こんどは**一番高い商品**が探せます。


### やってみよう（もうひとつ）

この商品カードには、カテゴリ（パン、飲み物、文具…）も入っています。
探しかたは値段のときと同じ。**タグと class の名前を変えるだけ**です。

In [ ]:
for item in items[:5]:
    name = item.find("h2", class_="item-name").text
    category = item.find("p", class_="item-category").text
    print(name, "/", category)

### 講師メモ

- 名前・値段・カテゴリで、`find` の書き方がまったく同じことを確認させる。
  **覚えることは増えていない**と言い切ってよい。
- `h2` → `p` にタグも変わっている点だけ、聞かれたら答える程度に触れる。
- `items[:5]` は「最初の5個だけ」。30行出すと画面が流れるので絞ってある。

## ステップ4：ほしいものだけ選ぶ

30個ぜんぶ並んでも、多すぎて選べません。
ここからは、取ったデータを**選んだり、ならべかえたり**します。

まずは「120円以下のものだけ」を出してみましょう。


### 4-1. 「もし〜なら」で選ぶ

`if` は「**もし〜なら**」。条件に合うときだけ `print` します。
`<=` は「以下」という意味です。


In [ ]:
for item in items:
    name = item.find("h2", class_="item-name").text
    price = int(item.find("span", class_="item-price").text)
    if price <= 120:
        print(name, price)


13行だけ出れば成功です。30個が13個にしぼれました。


### 4-2. 安い順にならべる

`append` は「リストの後ろに足す」、`sort()` は「小さい順にならべかえる」。
**値段を先に**書くのがコツで、その順にならんでくれます。


In [ ]:
prices = []

for item in items:
    name = item.find("h2", class_="item-name").text
    price = int(item.find("span", class_="item-price").text)
    prices.append((price, name))

prices.sort()


### 4-3. 安いほうから5つ

`prices[:5]` は「最初の5つだけ」という意味です。


In [ ]:
for price, name in prices[:5]:
    print(price, name)


### やってみよう

`prices[:5]` の `5` を `10` に変えてみましょう。
`prices.sort()` を `prices.sort(reverse=True)` にすると、こんどは高い順です。


### 講師メモ

- `if` は「もし〜なら」の一言でよい。`=` と `==` の違いは、聞かれたら答える程度に。
- 空欄は `<=`。プリント7章の記号表を見れば埋まる。**「以下」は `<=`、「未満」は `<`。**
- `prices.append((price, name))` のかっこ二重は写経でよい。
  「値段を先に書くと、値段の順にならぶ」ことだけ伝われば十分。
- 90円が2つ（使い捨てカイロ・消しゴム）あるが、並びは毎回同じになる。
  同じ値段のときは名前の順（文字コード順）。聞かれたら「値段が同じなら名前で決めてる」で十分。
- ここは**手が速い生徒とそうでない生徒の差が出やすい**。`4-3` まで行けば十分。


## ステップ5：カテゴリごとに数える

商品には、カテゴリ（パン・軽食・飲み物・文具・日用品）の名札も付いています。

```html
<p class="item-category">パン</p>
```

パンは平均いくらでしょう。飲み物は？ プログラムに数えさせます。


### 5-1. カテゴリごとに、合計と個数を数える

`{}` は **辞書**。「パン → 合計1090」のように、**名前で覚えておく入れもの**です。
`totals.get(cat, 0)` は「まだ無ければ0から始める」という意味。ここは写経で大丈夫。


In [ ]:
totals = {}
counts = {}

for item in items:
    cat = item.find("p", class_="item-category").text
    price = int(item.find("span", class_="item-price").text)
    totals[cat] = totals.get(cat, 0) + price
    counts[cat] = counts.get(cat, 0) + 1

print(counts)


### 5-2. 平均を出す

合計を個数で割れば平均です。`round(...)` は四捨五入。


In [ ]:
for cat in totals:
    print(cat, round(totals[cat] / counts[cat]))


日用品だけ、ずいぶん高く出ましたね。**本当に日用品は高いのでしょうか。**


### 5-3. 中身を見にいく

平均だけ見て決めつけず、日用品の中身を安い順に出してみます。


In [ ]:
for item in items:
    cat = item.find("p", class_="item-category").text
    if cat == "日用品":
        name = item.find("h2", class_="item-name").text
        price = int(item.find("span", class_="item-price").text)
        print(price, name)


980円の折りたたみ傘が1つ。これが平均を引き上げていました。

**平均は、1つの大きな値に引っぱられます。**
数字が出たら中身を見にいく。これはプログラムの話ではなく、データの話です。


### やってみよう

`"日用品"` を `"パン"` や `"飲み物"` に変えて、ほかのカテゴリも見てみましょう。


### 講師メモ

- 今日いちばん新しい文法（辞書）が出る場所。**説明しきろうとしないこと。**
  「名前で覚えておく入れもの」「`get(cat, 0)` は無ければ0から」で止める。
- 空欄は `item-category`。プリント6章の class 表にある。
- `5-2` の結果は パン182 / 軽食162 / 飲み物118 / 文具113 / **日用品285**。
  「日用品って高い？」と必ず問いかけてから `5-3` へ進む。
- `5-3` で 980円の折りたたみ傘が見える。**ここが今日いちばん学びのある場面。**

> 「平均が高いから日用品は高い、って言っていい？ 中身を見たら、傘が1本混ざってただけでした。
>  数字が出てきたら、中身を見にいく。これはプログラムの話じゃなくて、データの話です。」

- 時間が押していたら `5-3` だけでもやる。`5-1` `5-2` は講師機の画面で見せて進めてよい。


## ステップ6：詳細ページを辿る

一覧ページには**在庫**が載っていません。在庫は各商品の詳細ページにあります。
そこで「リンクをたどって、その先のページも読む」ということをします。


### 6-1. リンク先のアドレスを取り出す

`<a href="...">` の `href` の部分が、リンク先のアドレスです。


In [ ]:
for item in items:
    link = item.find("a", class_="item-link")
    print(link.get("href"))


`items/item01.html` のように、途中からのアドレスが出てきました。
`BASE_URL` とくっつけると、完全なアドレスになります。


### 6-2. 1つだけ詳細ページを開いてみる


In [ ]:
href = items[0].find("a", class_="item-link").get("href")

detail_res = requests.get(BASE_URL + href)
detail_res.encoding = "utf-8"
detail_soup = BeautifulSoup(detail_res.text, "html.parser")


In [ ]:
print(detail_soup.find("p", class_="item-stock").text)


### 講師メモ

- ここでやっていることはステップ1と同じ（取ってくる → 分解する → 探す）。
  「新しいことは何もしていない」と言い切ってしまってよい。
- `BASE_URL + href` の連結は、`BASE_URL` が `/` で終わっているから成立している。


### 6-3. 何個かまとめて見にいく

最後に、いくつかの商品の在庫をまとめて調べます。

`items[:5]` は「最初の5商品だけ」という意味です。
30商品ぜんぶ見にいくこともできますが、そのぶん相手にお願いする回数が増えます。

`time.sleep(1)` という行があります。これは**1秒待つ**という命令です。
相手のサーバーに一気に何回もお願いすると迷惑になるので、間を空けています。


In [ ]:
import time

for item in items[:5]:
    name = item.find("h2", class_="item-name").text
    href = item.find("a", class_="item-link").get("href")

    detail_res = requests.get(BASE_URL + href)
    detail_res.encoding = "utf-8"
    detail_soup = BeautifulSoup(detail_res.text, "html.parser")

    stock = detail_soup.find("p", class_="item-stock").text

    print(name, ":", stock)
    time.sleep(1)


### 講師メモ

- 5秒ほどかかる。「待たされている」という体感そのものが、
  次のマナーの話（負荷をかけるとはどういうことか）への導入になる。
- 「`[:5]` を外して30商品ぜんぶにしたら何秒かかる？」と口頭で問いかける（30秒）。
  **実際には流させない。** 待ち時間が伸びる体感が、そのまま相手側の負荷の話になる。
- `time.sleep(1)` を消したらどうなるか、を**口頭で**問いかけるだけにとどめる。
  実際に外部サイトへ試させないこと。


## ステップ7：本物のページに触ってみる（発展）

ここまでは、この授業のために用意した**きれいなページ**でした。
最後は本物、Wikipedia の記事から情報を取り出します。

やることは今までと同じ。取ってくる → 分解する → 探す。
違うのは、本物のページは**こちらの都合よく作られていない**ということだけです。


### 講師メモ

- 読みこむのは、**事前に1回だけ取得してリポジトリに置いてある保存版**（`samples/`）。
  生徒20人が本物の Wikipedia へ一斉に出ることはしない。ここでマナーの話を回収する。

> 「本物の Wikipedia を20人で一斉に叩くんじゃなくて、先生が前もって1回だけ取ってきて
>  置いてあります。さっきの話、そういうことです。」

- **ブラウザで本物の記事を開くのは自由**。人が読むぶんには普通の閲覧で、何も問題ない。
  「機械に何回もやらせる」ことだけが別の話、という線引きを見せる。
- 保存版は `<script>` `<style>` `<link>` を外してある。見た目は崩れるが、HTML の構造は本物のまま。


### 7-1. 保存してある記事を読みこむ

`samples/wikipedia-melonpan.html` に、Wikipedia の「メロンパン」の記事が置いてあります。
読みこみかたは、ステップ1とまったく同じです。


In [ ]:
WIKI_URL = BASE_URL + "samples/wikipedia-melonpan.html"

wiki_res = requests.get(WIKI_URL)
wiki_res.encoding = "utf-8"
wiki = BeautifulSoup(wiki_res.text, "html.parser")

print(wiki.title.text)


### 7-2. 記事のタイトルを取る

タイトルに「- Wikipedia」が付いてしまいました。記事名だけがほしいですね。

記事名は `<h1 id="firstHeading">` に入っています。
`id` は class と似ていますが、**ページに1つだけ**という印です。


In [ ]:
print(wiki.find("h1", id="firstHeading").text)


### 講師メモ

- `class_` と `id` の使い分けは1行で済ませる。
  「class は**同じ仲間が何個もある**印、id は**ページに1個しかない**印」。
- 30商品が全部 `class="item"` だったことを引きあいに出すと早い。
- `class_` にはアンダースコアが要るのに `id` には要らない。聞かれたら
  「`class` だけが Python の予約語とかぶっているから」とだけ答える。


### 7-3. 記事の見出しをならべる

大きな見出しは `<h2>` です。ステップ2と同じ、`find_all` の出番。


In [ ]:
for h in wiki.find_all("h2"):
    print(h.text)


1行目に「目次」が出てきましたね。これは記事の見出しではありません。

**本物のページは、こちらのほしいものだけを並べてはくれません。**
いらないものが混ざるのが普通です。


### やってみよう

`[1:]` を付けると「1つ目を飛ばす」という意味になります。「目次」を追い出してみましょう。


In [ ]:
for h in wiki.find_all("h2")[1:]:
    print(h.text)


### 講師メモ

- 「目次」が混ざるのは失敗ではなく**本物の姿**。ここは笑いどころにしてよい。
- 練習用サイトでは `class="item"` が用意されていたから一発だった、という対比を言う。

> 「さっきの購買部は、先生がみんなのために名前を付けておいたページです。
>  本物には、そんな親切はありません。」

- `[1:]` の説明は「1つ目を飛ばす」で止める。スライスの一般形には踏みこまない。


### 7-4. 書き出しの文章を取る

記事の1段落目は、最初の `<p>` に入っています。


In [ ]:
print(wiki.find("p").text)


`[1]` `[2]` という番号が混ざっています。これは**脚注の印**で、
`<sup>` というタグに入っています。

じゃまなので、消してから取り直しましょう。
`decompose()` は「そのタグをまるごと消す」という命令です。


In [ ]:
first = wiki.find("p")

for sup in first.find_all("sup"):
    sup.decompose()

print(first.text)


### 講師メモ

- 「取れた文字が、そのままでは使えない」という体験がこのステップの山。
  **本物のスクレイピングは、取ったあとの掃除が仕事の大半**だと一言添える。
- `decompose()` は覚えなくてよい。「消す命令がある」とだけ。
- 最後の一文「多くのメロンパンにはメロンが入っていない」で終わると場がなごむ。
- このセルを2回実行すると、`<sup>` はもう消えているので出力は同じ。エラーにはならない。


### 7-5. 相手が用意した入口を使う（講師機で1回だけ）

Wikipedia には、そもそも**データを渡すための入口（API）**が用意されています。
HTML をむしるのではなく、そちらに聞く方法です。


In [ ]:
# ★このセルは講師機で1回だけ実行する。生徒には実行させない。

headers = {
    # だれが何のために出しているリクエストかを名乗る。Wikimedia はこれを求めている
    "User-Agent": "CyberKoubaibu-Lesson/1.0 (https://github.com/yuracode/isshop) requests"
}
api_url = "https://ja.wikipedia.org/api/rest_v1/page/summary/メロンパン"

api_res = requests.get(api_url, headers=headers)
data = api_res.json()

print(data["title"])
print(data["extract"])


### 講師メモ

**このセルだけは外部（Wikipedia本体）にアクセスする。生徒には実行させず、講師画面で見せる。**

- 5-4 で苦労して掃除した文章が、掃除なしで返ってくる。`[1]` も付いていない。
  **「相手が用意した入口があるなら、そっちを使うほうが速いし、相手にもやさしい」**が結論。
- `headers` の `User-Agent` を画面で指させる。
  Wikimedia は自動アクセスに**連絡先入りの名乗り**を求めていて、`requests` の既定値
  （`python-requests/2.x`）は「使うな」と名指しされている側。
- **`headers` を外すと実際に `403 Forbidden` が返る**（2026-09-06 に確認済み）。
  余裕があれば、その場で `headers=headers` を消して1回だけ実行し、403 を見せるとよい。
  「名乗らない機械は入れてもらえない」という、いちばん分かりやすい実例になる。
- Wikipedia の robots.txt には
  "Friendly, low-speed bots are welcome viewing article pages" と書いてある。
  **「絶対だめ」ではなく「行儀よくならいい」**が正確なところ。ここは正直に伝える。
- 「じゃあ全部 API でいいのでは？」と聞かれたら
  「API を用意していないサイトのほうが多い。だから今日の技が要る」と答える。

### 7-5 を省略する場合

時間が足りなければ 5-4 で切ってよい。その場合も
**「保存版を読んだ」という事実だけは口頭で言う**こと（なぜ本物を叩かなかったのか）。


## おわりに

きょうやったことは、たった3つのくり返しでした。

1. `requests.get(...)` でページを取ってくる
2. `BeautifulSoup(...)` で分解する
3. `find` / `find_all` でほしい場所を探す

そのあとにやった「選ぶ・ならべる・数える」は、ふつうの計算です。
相手のサイトが変わっても、やることはこれだけです。
違うのは「どこを探すか」と、**相手に迷惑をかけない取り方をしているか**だけ。

おつかれさまでした。
